# Judge input

Turns `data/periods.jsonl` into batches for the LLM judge, one JSON object per line with `id`,
`source` and `text`, in the format `period_judge_prompt.md` describes. Ids are a hash of source
and text so they are stable; the order is shuffled with a fixed seed so every batch is a random
slice. `data/judge-input/index.json` maps each id back to its stores, paths and occurrences.

In [1]:
import hashlib
import json
import random
from pathlib import Path

BATCH_SIZE = 100
OUT_DIR = Path("data/judge-input")

index = {}
for line in Path("data/periods.jsonl").open(encoding="utf-8"):
    row = json.loads(line)
    id_ = hashlib.sha1(f"{row['source']}|{row['text']}".encode()).hexdigest()[:10]
    entry = index.setdefault(id_, {"source": row["source"], "text": row["text"], "stores": set(), "paths": set(), "occurrences": 0})
    entry["stores"].add(row["store"])
    entry["paths"].add(row["path"])
    entry["occurrences"] += row["n"]

ids = sorted(index)
random.Random(0).shuffle(ids)

OUT_DIR.mkdir(parents=True, exist_ok=True)
for old in OUT_DIR.glob("batch-*.jsonl"):
    old.unlink()
for b, start in enumerate(range(0, len(ids), BATCH_SIZE), 1):
    lines = [json.dumps({"id": id_, "source": index[id_]["source"], "text": index[id_]["text"]}, ensure_ascii=False) for id_ in ids[start : start + BATCH_SIZE]]
    (OUT_DIR / f"batch-{b:03}.jsonl").write_text("\n".join(lines) + "\n", encoding="utf-8")
json.dump(index, (OUT_DIR / "index.json").open("w", encoding="utf-8"), ensure_ascii=False, indent=1, default=sorted)

print(f"{len(ids)} distinct (source, text) pairs -> {b} batches in {OUT_DIR}")

75919 distinct (source, text) pairs -> 760 batches in data/judge-input
